In [1]:
from google.colab import auth
from google.cloud import bigquery

ModuleNotFoundError: No module named 'google.colab'

In [2]:
auth.authenticate_user()
client = bigquery.Client(project="careful-broker-438616-s1")

NameError: name 'auth' is not defined

In [ ]:
patients_table = """
SELECT 
subject_id,
gender,
dob
FROM `physionet-data.mimiciii_clinical.patients` pat
"""

admissions_table = """
SELECT 
subject_id,
admittime,
admission_type,
admission_location,
language,
religion,
marital_status,
ethnicity,
diagnosis AS consult_diagnosis,
hospital_expire_flag
FROM `physionet-data.mimiciii_clinical.admissions` adm
"""

diag_table = """
SELECT 
    subject_id,
    seq_num,
    diag.icd9_code AS diag_code,
    short_title AS diag_code_desc
FROM `physionet-data.mimiciii_clinical.diagnoses_icd` diag
INNER JOIN `physionet-data.mimiciii_clinical.d_icd_diagnoses` diagdesc
    ON diag.icd9 = diagdesc.icd9
"""


: 

In [ ]:
query_job = client.query(query)

df = query_job.result().to_dataframe()

: 

In [ ]:
patients_df = pd.DataFrame()  # Load data from the 'patients' table
admissions_df = pd.DataFrame()  # Load data from the 'admissions' table
diagnoses_df = pd.DataFrame()  # Load data from the 'diagnoses' table

In [ ]:
# Convert admittime to datetime
admissions_df['admittime'] = pd.to_datetime(admissions_df['admittime'])

# Sort by subject_id and admittime to order admissions by time (latest first)
admissions_df.sort_values(by=['subject_id', 'admittime'], ascending=[True, False], inplace=True)

# Create a count of previous admissions for each subject
admissions_df['previous_admits'] = admissions_df.groupby('subject_id').cumcount()

# Create columns for up to 3 previous consult diagnoses using shifts
admissions_df['previous_consult_diagnosis1'] = admissions_df.groupby('subject_id')['consult_diagnosis'].shift(-1)
admissions_df['previous_consult_diagnosis2'] = admissions_df.groupby('subject_id')['consult_diagnosis'].shift(-2)
admissions_df['previous_consult_diagnosis3'] = admissions_df.groupby('subject_id')['consult_diagnosis'].shift(-3)

# Filter to keep only the most recent admission per subject_id
most_recent_admissions_df = admissions_df.drop_duplicates(subset='subject_id', keep='first')

# Keep only necessary columns
most_recent_admissions_df = most_recent_admissions_df[['subject_id', 'admittime', 'admission_type', 'admission_location',
                                                       'language', 'religion', 'marital_status', 'ethnicity',
                                                       'consult_diagnosis', 'previous_admits', 'previous_consult_diagnosis1',
                                                       'previous_consult_diagnosis2', 'previous_consult_diagnosis3',
                                                       'hospital_expire_flag']]
